In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('MyApplication').getOrCreate()
sc=spark.sparkContext

#**1. Department-wise Salary Analysis**

You are given employee data containing employee_id, employee_name, department, and salary. Using RDD operations, calculate the total salary and average salary for each department. Display only departments whose average salary is greater than ₹50,000 and sort them by average salary in descending order.

Concepts: map(), reduceByKey(), mapValues(), filter(), sortBy()

In [ ]:
data = [
    ("EMP001", "Aarav Sharma", "IT", 65000),
    ("EMP002", "Priya Patel", "HR", 58000),
    ("EMP003", "Rohan Mehta", "Finance", 72000),
    ("EMP004", "Ananya Singh", "Marketing", 61000),
    ("EMP005", "Vikram Das", "IT", 78000),
    ("EMP006", "Sneha Nair", "HR", 55000),
    ("EMP007", "Arjun Verma", "Finance", 69000),
    ("EMP008", "Kavya Rao", "Sales", 63000),
    ("EMP009", "Rahul Gupta", "Marketing", 59000),
    ("EMP010", "Neha Kapoor", "Sales", 67000)
]

rdd = sc.parallelize(data)

In [ ]:
reduced_rdd=rdd.map(lambda x:(x[2],x[3]))
reduced_rdd.collect()

[('IT', 65000),
 ('HR', 58000),
 ('Finance', 72000),
 ('Marketing', 61000),
 ('IT', 78000),
 ('HR', 55000),
 ('Finance', 69000),
 ('Sales', 63000),
 ('Marketing', 59000),
 ('Sales', 67000)]

Total salary for each department

In [ ]:
total_salary=reduced_rdd.reduceByKey(lambda x,y:x+y)
print("Total salary for each department:")
total_salary.collect()

Total salary for each department:


[('IT', 143000),
 ('HR', 113000),
 ('Finance', 141000),
 ('Sales', 130000),
 ('Marketing', 120000)]

Average salary of each department

In [ ]:
dept_salary = rdd.map(lambda x: (x[2], (int(x[3]), 1)))
result=dept_salary.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
avg_salary = result.mapValues(lambda x: x[0] / x[1])
print("Average salary for each department:")
avg_salary.collect()


Average salary for each department:


[('IT', 71500.0),
 ('HR', 56500.0),
 ('Finance', 70500.0),
 ('Sales', 65000.0),
 ('Marketing', 60000.0)]

map() → Modify entire record (key and value)

mapValues() → Modify only value, keep key same

Display only departments whose average salary is greater than ₹50,000 and sort them by average salary in descending order.


In [ ]:
dept=avg_salary.filter(lambda x:x[1]>=50000).sortBy(lambda x:x[1],ascending=False)
dept.collect()

[('IT', 71500.0),
 ('Finance', 70500.0),
 ('Sales', 65000.0),
 ('Marketing', 60000.0),
 ('HR', 56500.0)]

#***2. Customer Purchase Summary***

You are given transaction data containing transaction_id, customer_id, product, amount, and status. Filter only "Completed" transactions and calculate the total purchase amount for each customer. Display the top 5 customers based on their total spending.

Concepts: filter(), map(), reduceByKey(), sortBy(), take()

In [ ]:
data = [
    ("T001", "C101", "Laptop", 55000, "Completed"),
    ("T002", "C102", "Smartphone", 32000, "Pending"),
    ("T003", "C103", "Headphones", 2500, "Completed"),
    ("T004", "C104", "Keyboard", 1800, "Failed"),
    ("T005", "C105", "Monitor", 15000, "Completed"),
    ("T006", "C106", "Mouse", 1200, "Pending"),
    ("T007", "C107", "Tablet", 28000, "Completed"),
    ("T008", "C108", "Smartwatch", 7500, "Cancelled"),
    ("T009", "C109", "Printer", 12500, "Completed"),
    ("T010", "C110", "Webcam", 4500, "Failed"),
    ("T011", "C101", "Keyboard", 2200, "Completed"),
    ("T012", "C102", "Monitor", 18000, "Completed"),
    ("T013", "C103", "Mouse", 1500, "Pending"),
    ("T014", "C104", "Laptop", 62000, "Completed"),
    ("T015", "C105", "Smartphone", 35000, "Failed"),
    ("T016", "C106", "Tablet", 26000, "Completed"),
    ("T017", "C107", "Headphones", 3000, "Completed"),
    ("T018", "C108", "Printer", 14000, "Pending"),
    ("T019", "C109", "Webcam", 5000, "Completed"),
    ("T020", "C110", "Smartwatch", 8000, "Completed")
]

rdd = sc.parallelize(data)

In [ ]:
comp=rdd.filter(lambda x:x[4]=='Completed')
comp.collect()

[('T001', 'C101', 'Laptop', 55000, 'Completed'),
 ('T003', 'C103', 'Headphones', 2500, 'Completed'),
 ('T005', 'C105', 'Monitor', 15000, 'Completed'),
 ('T007', 'C107', 'Tablet', 28000, 'Completed'),
 ('T009', 'C109', 'Printer', 12500, 'Completed'),
 ('T011', 'C101', 'Keyboard', 2200, 'Completed'),
 ('T012', 'C102', 'Monitor', 18000, 'Completed'),
 ('T014', 'C104', 'Laptop', 62000, 'Completed'),
 ('T016', 'C106', 'Tablet', 26000, 'Completed'),
 ('T017', 'C107', 'Headphones', 3000, 'Completed'),
 ('T019', 'C109', 'Webcam', 5000, 'Completed'),
 ('T020', 'C110', 'Smartwatch', 8000, 'Completed')]

In [ ]:
reduced_comp=comp.map(lambda x:(x[1],x[3]))
reduced_comp.collect()

[('C101', 55000),
 ('C103', 2500),
 ('C105', 15000),
 ('C107', 28000),
 ('C109', 12500),
 ('C101', 2200),
 ('C102', 18000),
 ('C104', 62000),
 ('C106', 26000),
 ('C107', 3000),
 ('C109', 5000),
 ('C110', 8000)]

In [ ]:
total_amt=reduced_comp.reduceByKey(lambda x,y:x+y)
total_amt.collect()

[('C103', 2500),
 ('C105', 15000),
 ('C109', 17500),
 ('C102', 18000),
 ('C106', 26000),
 ('C110', 8000),
 ('C101', 57200),
 ('C107', 31000),
 ('C104', 62000)]

In [ ]:
result=total_amt.sortBy(lambda x:x[1],ascending=False).take(5)
print("Top 5 customers with the highest total purchase amount:",result)

Top 5 customers with the highest total purchase amount: [('C104', 62000), ('C101', 57200), ('C107', 31000), ('C106', 26000), ('C102', 18000)]


#***3. Word Frequency Analysis***

You are given an RDD containing multiple sentences. Convert all words to lowercase, split the sentences into individual words, remove words having fewer than four characters, and calculate the frequency of each remaining word. Display the 10 most frequently occurring words.

Concepts: flatMap(), map(), filter(), reduceByKey(), takeOrdered()

In [ ]:
data = [
    "Spark is a powerful data processing framework",
    "Python is easy to learn and use",
    "RDD is a fundamental concept in Spark",
    "Big data requires efficient processing techniques",
    "Spark supports distributed computing",
    "Machine learning helps solve complex problems",
    "Data engineers work with large datasets",
    "Apache Spark can process data quickly",
    "Python is widely used for data analysis",
    "RDDs are immutable distributed collections"
]

rdd = sc.parallelize(data)

In [ ]:
words=rdd.flatMap(lambda x:x.split())
lower_words=words.map(lambda x:x.lower())
lower_words.collect()

['spark',
 'is',
 'a',
 'powerful',
 'data',
 'processing',
 'framework',
 'python',
 'is',
 'easy',
 'to',
 'learn',
 'and',
 'use',
 'rdd',
 'is',
 'a',
 'fundamental',
 'concept',
 'in',
 'spark',
 'big',
 'data',
 'requires',
 'efficient',
 'processing',
 'techniques',
 'spark',
 'supports',
 'distributed',
 'computing',
 'machine',
 'learning',
 'helps',
 'solve',
 'complex',
 'problems',
 'data',
 'engineers',
 'work',
 'with',
 'large',
 'datasets',
 'apache',
 'spark',
 'can',
 'process',
 'data',
 'quickly',
 'python',
 'is',
 'widely',
 'used',
 'for',
 'data',
 'analysis',
 'rdds',
 'are',
 'immutable',
 'distributed',
 'collections']

In [ ]:
lower_words=lower_words.filter(lambda x:len(x)>4)
lower_words.collect()

['spark',
 'powerful',
 'processing',
 'framework',
 'python',
 'learn',
 'fundamental',
 'concept',
 'spark',
 'requires',
 'efficient',
 'processing',
 'techniques',
 'spark',
 'supports',
 'distributed',
 'computing',
 'machine',
 'learning',
 'helps',
 'solve',
 'complex',
 'problems',
 'engineers',
 'large',
 'datasets',
 'apache',
 'spark',
 'process',
 'quickly',
 'python',
 'widely',
 'analysis',
 'immutable',
 'distributed',
 'collections']

In [ ]:
count_words=lower_words.map(lambda x:(x,1))
reduced_words=count_words.reduceByKey(lambda x,y:x+y).takeOrdered(10,key=lambda x:-x[1])
reduced_words

[('spark', 4),
 ('python', 2),
 ('distributed', 2),
 ('processing', 2),
 ('powerful', 1),
 ('framework', 1),
 ('learn', 1),
 ('requires', 1),
 ('efficient', 1),
 ('supports', 1)]

#***4. Customer and Order Join***

Create two RDDs: customers containing customer_id, customer_name, and city, and orders containing order_id, customer_id, and order_amount. Join the two RDDs using customer_id and generate an output containing:

customer_id, customer_name, city, order_amount

Also identify customers who have not placed any orders.

Concepts: Key-value RDD, map(), join(), leftOuterJoin(), filter()

In [ ]:
customers = sc.parallelize([
    (101, "Rahul", "Delhi"),
    (102, "Priya", "Mumbai"),
    (103, "Amit", "Bangalore"),
    (104, "Sneha", "Hyderabad"),
    (105, "Rohit", "Chennai")
])

orders = sc.parallelize([
    (1001, 101, 5500),
    (1002, 102, 3200),
    (1003, 101, 7500),
    (1005, 104, 6800),
    (1006, 105, 2500),
    (1007, 102, 4500)
])

print("Customers:")
print(customers.collect())

print("\nOrders:")
print(orders.collect())

Customers:
[(101, 'Rahul', 'Delhi'), (102, 'Priya', 'Mumbai'), (103, 'Amit', 'Bangalore'), (104, 'Sneha', 'Hyderabad'), (105, 'Rohit', 'Chennai')]

Orders:
[(1001, 101, 5500), (1002, 102, 3200), (1003, 101, 7500), (1005, 104, 6800), (1006, 105, 2500), (1007, 102, 4500)]


In [ ]:
customers_rdd=customers.map(lambda x:(x[0],(x[1],x[2])))
customers_rdd.collect()

[(101, ('Rahul', 'Delhi')),
 (102, ('Priya', 'Mumbai')),
 (103, ('Amit', 'Bangalore')),
 (104, ('Sneha', 'Hyderabad')),
 (105, ('Rohit', 'Chennai'))]

In [ ]:
orders_rdd=orders.map(lambda x:(x[1],(x[2])))
orders_rdd.collect()

[(101, 5500), (102, 3200), (101, 7500), (104, 6800), (105, 2500), (102, 4500)]

In [ ]:
result=customers_rdd.leftOuterJoin(orders_rdd)
result.collect()

[(104, (('Sneha', 'Hyderabad'), 6800)),
 (101, (('Rahul', 'Delhi'), 5500)),
 (101, (('Rahul', 'Delhi'), 7500)),
 (105, (('Rohit', 'Chennai'), 2500)),
 (102, (('Priya', 'Mumbai'), 3200)),
 (102, (('Priya', 'Mumbai'), 4500)),
 (103, (('Amit', 'Bangalore'), None))]

In [ ]:
no_order=result.filter(lambda x:x[1][1]==None)
names = no_order.map(lambda x: x[1][0][0])
names.collect()

['Amit']

#***5. Product-wise Sales Analysis***

You are given sales data containing order_id, product_name, category, quantity, and unit_price. Calculate:

sales_amount = quantity × unit_price

Then calculate the total sales amount for each product and display products whose total sales exceed ₹10,000, sorted from highest to lowest sales.

Concepts: map(), reduceByKey(), filter(), sortBy()

In [ ]:
sales = [
    (1001, "Laptop", "Electronics", 2, 55000),
    (1002, "Mouse", "Electronics", 5, 800),
    (1003, "Keyboard", "Electronics", 3, 1500),
    (1004, "Office Chair", "Furniture", 2, 7500),
    (1005, "Desk", "Furniture", 1, 12000),
    (1006, "Headphones", "Electronics", 4, 2500),
    (1007, "Monitor", "Electronics", 2, 18000),
    (1008, "Notebook", "Stationery", 10, 120),
    (1009, "Pen", "Stationery", 20, 50),
    (1010, "Backpack", "Accessories", 3, 1800)
]

sales_rdd = sc.parallelize(sales)

sales_rdd.collect()

[(1001, 'Laptop', 'Electronics', 2, 55000),
 (1002, 'Mouse', 'Electronics', 5, 800),
 (1003, 'Keyboard', 'Electronics', 3, 1500),
 (1004, 'Office Chair', 'Furniture', 2, 7500),
 (1005, 'Desk', 'Furniture', 1, 12000),
 (1006, 'Headphones', 'Electronics', 4, 2500),
 (1007, 'Monitor', 'Electronics', 2, 18000),
 (1008, 'Notebook', 'Stationery', 10, 120),
 (1009, 'Pen', 'Stationery', 20, 50),
 (1010, 'Backpack', 'Accessories', 3, 1800)]

In [ ]:
sales_rdd_new=sales_rdd.map(lambda x:(x[2],x[3]*x[4]))
sales_rdd_new.collect()

[('Electronics', 110000),
 ('Electronics', 4000),
 ('Electronics', 4500),
 ('Furniture', 15000),
 ('Furniture', 12000),
 ('Electronics', 10000),
 ('Electronics', 36000),
 ('Stationery', 1200),
 ('Stationery', 1000),
 ('Accessories', 5400)]

In [ ]:
product_sales=sales_rdd_new.reduceByKey(lambda x,y:x+y).filter(lambda x:x[1]>10000).sortBy(lambda x:x[1],ascending=False)
product_sales.collect()

[('Electronics', 164500), ('Furniture', 27000)]

#***6. Find the Highest-Paid Employee in Each Department***

You are given employee records containing:

employee_id
employee_name
department
salary

Using RDD operations, find the employee with the highest salary in each department. Your final output should contain:

department, employee_name, salary

Do not use DataFrames or Spark SQL.

Concepts: map(), key-value RDD, reduceByKey()

In [ ]:
employees = [
    (101, "Rahul", "IT", 65000),
    (102, "Priya", "HR", 55000),
    (103, "Amit", "Finance", 72000),
    (104, "Sneha", "IT", 80000),
    (105, "Rohit", "Marketing", 60000),
    (106, "Neha", "HR", 58000),
    (107, "Vikash", "Finance", 75000),
    (108, "Anjali", "IT", 90000),
    (109, "Karan", "Marketing", 62000),
    (110, "Pooja", "Finance", 68000)
]

employee_rdd = sc.parallelize(employees)

employee_rdd.collect()

[(101, 'Rahul', 'IT', 65000),
 (102, 'Priya', 'HR', 55000),
 (103, 'Amit', 'Finance', 72000),
 (104, 'Sneha', 'IT', 80000),
 (105, 'Rohit', 'Marketing', 60000),
 (106, 'Neha', 'HR', 58000),
 (107, 'Vikash', 'Finance', 75000),
 (108, 'Anjali', 'IT', 90000),
 (109, 'Karan', 'Marketing', 62000),
 (110, 'Pooja', 'Finance', 68000)]

In [ ]:
employee_rdd_new=employee_rdd.map(lambda x:(x[2],(x[1],x[3])))
employee_rdd_new.collect()

[('IT', ('Rahul', 65000)),
 ('HR', ('Priya', 55000)),
 ('Finance', ('Amit', 72000)),
 ('IT', ('Sneha', 80000)),
 ('Marketing', ('Rohit', 60000)),
 ('HR', ('Neha', 58000)),
 ('Finance', ('Vikash', 75000)),
 ('IT', ('Anjali', 90000)),
 ('Marketing', ('Karan', 62000)),
 ('Finance', ('Pooja', 68000))]

In [ ]:
highest_salary=employee_rdd_new.reduceByKey(lambda x,y:x if x[1]>y[1] else y)
highest_salary.collect()

[('IT', ('Anjali', 90000)),
 ('HR', ('Neha', 58000)),
 ('Finance', ('Vikash', 75000)),
 ('Marketing', ('Karan', 62000))]

#***7. Order Status Analysis***


You are given e-commerce order data containing order_id, customer_id, amount, and order_status. Calculate the number of orders for each status, such as:

Completed
Cancelled
Pending
Returned

Then determine what percentage of the total orders belongs to each status.

Example:

Completed   65%
Cancelled   15%
Pending     12%
Returned     8%

Concepts: map(), reduceByKey(), count(), mapValues()

In [ ]:
orders = [
    (1001, 101, 5500, "Completed"),
    (1002, 102, 3200, "Pending"),
    (1003, 103, 7500, "Completed"),
    (1004, 104, 4200, "Cancelled"),
    (1005, 105, 6800, "Completed"),
    (1006, 106, 2500, "Returned"),
    (1007, 107, 4500, "Completed"),
    (1008, 108, 3900, "Pending"),
    (1009, 109, 8200, "Cancelled"),
    (1010, 110, 6000, "Completed"),
    (1011, 101, 7200, "Returned"),
    (1012, 102, 1800, "Completed"),
    (1013, 103, 9500, "Pending"),
    (1014, 104, 3600, "Returned"),
    (1015, 105, 4800, "Completed"),
    (1016, 106, 2500, "Cancelled"),
    (1017, 107, 6700, "Pending"),
    (1018, 108, 8900, "Completed"),
    (1019, 109, 3100, "Returned"),
    (1020, 110, 5500, "Completed")
]

orders_rdd = sc.parallelize(orders)

orders_rdd.collect()

[(1001, 101, 5500, 'Completed'),
 (1002, 102, 3200, 'Pending'),
 (1003, 103, 7500, 'Completed'),
 (1004, 104, 4200, 'Cancelled'),
 (1005, 105, 6800, 'Completed'),
 (1006, 106, 2500, 'Returned'),
 (1007, 107, 4500, 'Completed'),
 (1008, 108, 3900, 'Pending'),
 (1009, 109, 8200, 'Cancelled'),
 (1010, 110, 6000, 'Completed'),
 (1011, 101, 7200, 'Returned'),
 (1012, 102, 1800, 'Completed'),
 (1013, 103, 9500, 'Pending'),
 (1014, 104, 3600, 'Returned'),
 (1015, 105, 4800, 'Completed'),
 (1016, 106, 2500, 'Cancelled'),
 (1017, 107, 6700, 'Pending'),
 (1018, 108, 8900, 'Completed'),
 (1019, 109, 3100, 'Returned'),
 (1020, 110, 5500, 'Completed')]

In [ ]:
no_orders=orders_rdd.map(lambda x:(x[3],1))
no_orders.collect()

[('Completed', 1),
 ('Pending', 1),
 ('Completed', 1),
 ('Cancelled', 1),
 ('Completed', 1),
 ('Returned', 1),
 ('Completed', 1),
 ('Pending', 1),
 ('Cancelled', 1),
 ('Completed', 1),
 ('Returned', 1),
 ('Completed', 1),
 ('Pending', 1),
 ('Returned', 1),
 ('Completed', 1),
 ('Cancelled', 1),
 ('Pending', 1),
 ('Completed', 1),
 ('Returned', 1),
 ('Completed', 1)]

In [ ]:
no_orders_new=no_orders.reduceByKey(lambda x,y:x+y)
no_orders_new.collect()

[('Cancelled', 3), ('Completed', 9), ('Pending', 4), ('Returned', 4)]

In [ ]:
total_orders=no_orders.count()
total_orders

20

In [ ]:
perc_orders=no_orders_new.mapValues(lambda x:f'{(x/total_orders)*100:.0f}%')
perc_orders.collect()

[('Cancelled', '15%'),
 ('Completed', '45%'),
 ('Pending', '20%'),
 ('Returned', '20%')]

#***8. Student Performance Analysis***


You are given student marks containing:

student_id, student_name, subject, marks

A student can have multiple records for different subjects. Calculate the average marks for each student. Keep only students whose average is at least 60, and display the results in descending order of average marks.

Concepts: map(), reduceByKey(), mapValues(), filter(), sortBy()

In [ ]:
student_marks = [
    (101, "Rahul", "Mathematics", 85),
    (101, "Rahul", "Physics", 78),
    (101, "Rahul", "Chemistry", 82),

    (102, "Priya", "Mathematics", 92),
    (102, "Priya", "Physics", 88),
    (102, "Priya", "Chemistry", 90),

    (103, "Amit", "Mathematics", 75),
    (103, "Amit", "Physics", 80),
    (103, "Amit", "Chemistry", 72),

    (104, "Sneha", "Mathematics", 95),
    (104, "Sneha", "Physics", 91),
    (104, "Sneha", "Chemistry", 94),

    (105, "Rohit", "Mathematics", 68),
    (105, "Rohit", "Physics", 74),
    (105, "Rohit", "Chemistry", 70),

    (106, "Neha", "Mathematics", 88),
    (106, "Neha", "Physics", 85),
    (106, "Neha", "Chemistry", 89),

    (107, "Karan", "Mathematics", 79),
    (107, "Karan", "Physics", 83),
    (107, "Karan", "Chemistry", 77),

    (108, "Anjali", "Mathematics", 91),
    (108, "Anjali", "Physics", 87),
    (108, "Anjali", "Chemistry", 93)
]

student_rdd = sc.parallelize(student_marks)

student_rdd.collect()

[(101, 'Rahul', 'Mathematics', 85),
 (101, 'Rahul', 'Physics', 78),
 (101, 'Rahul', 'Chemistry', 82),
 (102, 'Priya', 'Mathematics', 92),
 (102, 'Priya', 'Physics', 88),
 (102, 'Priya', 'Chemistry', 90),
 (103, 'Amit', 'Mathematics', 75),
 (103, 'Amit', 'Physics', 80),
 (103, 'Amit', 'Chemistry', 72),
 (104, 'Sneha', 'Mathematics', 95),
 (104, 'Sneha', 'Physics', 91),
 (104, 'Sneha', 'Chemistry', 94),
 (105, 'Rohit', 'Mathematics', 68),
 (105, 'Rohit', 'Physics', 74),
 (105, 'Rohit', 'Chemistry', 70),
 (106, 'Neha', 'Mathematics', 88),
 (106, 'Neha', 'Physics', 85),
 (106, 'Neha', 'Chemistry', 89),
 (107, 'Karan', 'Mathematics', 79),
 (107, 'Karan', 'Physics', 83),
 (107, 'Karan', 'Chemistry', 77),
 (108, 'Anjali', 'Mathematics', 91),
 (108, 'Anjali', 'Physics', 87),
 (108, 'Anjali', 'Chemistry', 93)]

In [ ]:
student_details=student_rdd.map(lambda x:(x[0],(x[1],x[2],x[3],1)))
student_details.collect()

[(101, ('Rahul', 'Mathematics', 85, 1)),
 (101, ('Rahul', 'Physics', 78, 1)),
 (101, ('Rahul', 'Chemistry', 82, 1)),
 (102, ('Priya', 'Mathematics', 92, 1)),
 (102, ('Priya', 'Physics', 88, 1)),
 (102, ('Priya', 'Chemistry', 90, 1)),
 (103, ('Amit', 'Mathematics', 75, 1)),
 (103, ('Amit', 'Physics', 80, 1)),
 (103, ('Amit', 'Chemistry', 72, 1)),
 (104, ('Sneha', 'Mathematics', 95, 1)),
 (104, ('Sneha', 'Physics', 91, 1)),
 (104, ('Sneha', 'Chemistry', 94, 1)),
 (105, ('Rohit', 'Mathematics', 68, 1)),
 (105, ('Rohit', 'Physics', 74, 1)),
 (105, ('Rohit', 'Chemistry', 70, 1)),
 (106, ('Neha', 'Mathematics', 88, 1)),
 (106, ('Neha', 'Physics', 85, 1)),
 (106, ('Neha', 'Chemistry', 89, 1)),
 (107, ('Karan', 'Mathematics', 79, 1)),
 (107, ('Karan', 'Physics', 83, 1)),
 (107, ('Karan', 'Chemistry', 77, 1)),
 (108, ('Anjali', 'Mathematics', 91, 1)),
 (108, ('Anjali', 'Physics', 87, 1)),
 (108, ('Anjali', 'Chemistry', 93, 1))]

In [ ]:
reduced_details=student_details.reduceByKey(lambda x,y:(x[0],x[1],x[2]+y[2],x[3]+y[3]))
reduced_details.collect()

[(102, ('Priya', 'Mathematics', 270, 3)),
 (104, ('Sneha', 'Mathematics', 280, 3)),
 (106, ('Neha', 'Mathematics', 262, 3)),
 (108, ('Anjali', 'Mathematics', 271, 3)),
 (101, ('Rahul', 'Mathematics', 245, 3)),
 (103, ('Amit', 'Mathematics', 227, 3)),
 (105, ('Rohit', 'Mathematics', 212, 3)),
 (107, ('Karan', 'Mathematics', 239, 3))]

In [ ]:
avg_details=reduced_details.mapValues(lambda x:f'{(x[2]/x[3]):.2f}').filter(lambda x:float(x[1])>=60).sortBy(lambda x:float(x[1]),ascending=False)
avg_details.collect()

[(104, '93.33'),
 (108, '90.33'),
 (102, '90.00'),
 (106, '87.33'),
 (101, '81.67'),
 (107, '79.67'),
 (103, '75.67'),
 (105, '70.67')]

#***9. Customer Purchase Category Analysis***

You are given transaction records containing:

transaction_id
customer_id
category
amount

Calculate the total amount spent by each customer in each category.

The expected result should look like:

(101, Electronics) → 25000
(101, Grocery)     → 5000
(102, Electronics) → 18000

After calculating the totals, display only customer-category combinations where spending exceeds ₹10,000.

Concepts: Composite keys, map(), reduceByKey(), filter()

In [ ]:
transactions = [
    (1001, 101, "Electronics", 55000),
    (1002, 102, "Clothing", 2500),
    (1003, 101, "Groceries", 1800),
    (1004, 103, "Electronics", 32000),
    (1005, 104, "Furniture", 12500),
    (1006, 102, "Groceries", 3500),
    (1007, 105, "Clothing", 4200),
    (1008, 103, "Electronics", 18500),
    (1009, 101, "Furniture", 9000),
    (1010, 106, "Groceries", 2200),
    (1011, 104, "Electronics", 45000),
    (1012, 105, "Clothing", 3800),
    (1013, 107, "Furniture", 15000),
    (1014, 102, "Electronics", 28000),
    (1015, 106, "Clothing", 5200),
    (1016, 103, "Groceries", 4100),
    (1017, 107, "Electronics", 36000),
    (1018, 101, "Clothing", 2900),
    (1019, 104, "Groceries", 2700),
    (1020, 105, "Furniture", 11000)
]

transaction_rdd = sc.parallelize(transactions)

transaction_rdd.collect()

[(1001, 101, 'Electronics', 55000),
 (1002, 102, 'Clothing', 2500),
 (1003, 101, 'Groceries', 1800),
 (1004, 103, 'Electronics', 32000),
 (1005, 104, 'Furniture', 12500),
 (1006, 102, 'Groceries', 3500),
 (1007, 105, 'Clothing', 4200),
 (1008, 103, 'Electronics', 18500),
 (1009, 101, 'Furniture', 9000),
 (1010, 106, 'Groceries', 2200),
 (1011, 104, 'Electronics', 45000),
 (1012, 105, 'Clothing', 3800),
 (1013, 107, 'Furniture', 15000),
 (1014, 102, 'Electronics', 28000),
 (1015, 106, 'Clothing', 5200),
 (1016, 103, 'Groceries', 4100),
 (1017, 107, 'Electronics', 36000),
 (1018, 101, 'Clothing', 2900),
 (1019, 104, 'Groceries', 2700),
 (1020, 105, 'Furniture', 11000)]

In [ ]:
reduced_transaction=transaction_rdd.map(lambda x:((x[1],x[2]),x[3]))
reduced_transaction.collect()

[((101, 'Electronics'), 55000),
 ((102, 'Clothing'), 2500),
 ((101, 'Groceries'), 1800),
 ((103, 'Electronics'), 32000),
 ((104, 'Furniture'), 12500),
 ((102, 'Groceries'), 3500),
 ((105, 'Clothing'), 4200),
 ((103, 'Electronics'), 18500),
 ((101, 'Furniture'), 9000),
 ((106, 'Groceries'), 2200),
 ((104, 'Electronics'), 45000),
 ((105, 'Clothing'), 3800),
 ((107, 'Furniture'), 15000),
 ((102, 'Electronics'), 28000),
 ((106, 'Clothing'), 5200),
 ((103, 'Groceries'), 4100),
 ((107, 'Electronics'), 36000),
 ((101, 'Clothing'), 2900),
 ((104, 'Groceries'), 2700),
 ((105, 'Furniture'), 11000)]

In [ ]:
category_transaction=reduced_transaction.reduceByKey(lambda x,y:x+y).filter(lambda x:x[1]>10000)
category_transaction.collect()

[((101, 'Electronics'), 55000),
 ((103, 'Electronics'), 50500),
 ((104, 'Furniture'), 12500),
 ((107, 'Electronics'), 36000),
 ((104, 'Electronics'), 45000),
 ((107, 'Furniture'), 15000),
 ((102, 'Electronics'), 28000),
 ((105, 'Furniture'), 11000)]

In [ ]:
result=category_transaction.map(lambda x:f'{(x[0][0],x[0][1])} -> {x[1]}')
result.collect()

["(101, 'Electronics') -> 55000",
 "(103, 'Electronics') -> 50500",
 "(104, 'Furniture') -> 12500",
 "(107, 'Electronics') -> 36000",
 "(104, 'Electronics') -> 45000",
 "(107, 'Furniture') -> 15000",
 "(102, 'Electronics') -> 28000",
 "(105, 'Furniture') -> 11000"]

#***10. Complete E-commerce RDD Analysis***

You are provided with three RDDs:

Customers

customer_id, customer_name, city

Orders

order_id, customer_id, order_date, status

Order Items

order_id, product_name, quantity, unit_price

Using only RDD operations:

Filter only "Completed" orders.
Calculate item_value = quantity × unit_price.
Calculate the total value of each order.
Join the order totals with the completed orders.
Join the result with customer information.
Calculate the total spending of each customer.
Keep customers whose total spending exceeds ₹10,000.
Sort customers by total spending in descending order.
Display the top 5 customers.
Produce the final output as:
Customer_ID | Customer_Name | City | Total_Spending

Concepts: filter(), map(), reduceByKey(), join(), composite processing, sortBy(), take()

In [ ]:
customers = [
    (101, "Rahul", "Delhi"),
    (102, "Priya", "Mumbai"),
    (103, "Amit", "Bangalore"),
    (104, "Sneha", "Hyderabad"),
    (105, "Rohit", "Chennai"),
    (106, "Neha", "Pune"),
    (107, "Karan", "Kolkata"),
    (108, "Anjali", "Bhubaneswar")
]

customers_rdd = sc.parallelize(customers)

In [ ]:
orders = [
    (1001, 101, "2026-09-01", "Completed"),
    (1002, 102, "2026-09-02", "Pending"),
    (1003, 101, "2026-09-03", "Completed"),
    (1004, 103, "2026-09-04", "Cancelled"),
    (1005, 104, "2026-09-05", "Completed"),
    (1006, 105, "2026-09-06", "Returned"),
    (1007, 102, "2026-09-07", "Completed"),
    (1008, 106, "2026-09-08", "Completed"),
    (1009, 107, "2026-09-09", "Pending"),
    (1010, 108, "2026-09-10", "Completed"),
    (1011, 103, "2026-09-11", "Completed"),
    (1012, 104, "2026-09-12", "Pending")
]

orders_rdd = sc.parallelize(orders)

In [ ]:
order_items = [
    (1001, "Laptop", 1, 55000),
    (1001, "Mouse", 2, 800),
    (1002, "Headphones", 1, 2500),
    (1003, "Keyboard", 1, 1500),
    (1003, "Monitor", 1, 18000),
    (1004, "Smartphone", 1, 32000),
    (1005, "Office Chair", 2, 7500),
    (1006, "Backpack", 1, 1800),
    (1007, "Tablet", 1, 22000),
    (1008, "Printer", 1, 12000),
    (1009, "Desk", 1, 15000),
    (1010, "Smartwatch", 2, 5000),
    (1011, "Laptop", 1, 65000),
    (1012, "Keyboard", 2, 1500)
]

order_items_rdd = sc.parallelize(order_items)

In [ ]:
reduced_orders=orders_rdd.filter(lambda x:x[3]=='Completed')
reduced_orders.collect()

[(1001, 101, '2026-09-01', 'Completed'),
 (1003, 101, '2026-09-03', 'Completed'),
 (1005, 104, '2026-09-05', 'Completed'),
 (1007, 102, '2026-09-07', 'Completed'),
 (1008, 106, '2026-09-08', 'Completed'),
 (1010, 108, '2026-09-10', 'Completed'),
 (1011, 103, '2026-09-11', 'Completed')]

In [ ]:
updated_items=order_items_rdd.map(lambda x:(x[0],x[2]*x[3]))
updated_items.collect()

[(1001, 55000),
 (1001, 1600),
 (1002, 2500),
 (1003, 1500),
 (1003, 18000),
 (1004, 32000),
 (1005, 15000),
 (1006, 1800),
 (1007, 22000),
 (1008, 12000),
 (1009, 15000),
 (1010, 10000),
 (1011, 65000),
 (1012, 3000)]

In [ ]:
reduced_items=updated_items.reduceByKey(lambda x,y:x+y)
reduced_items.collect()

[(1002, 2500),
 (1004, 32000),
 (1006, 1800),
 (1008, 12000),
 (1010, 10000),
 (1012, 3000),
 (1001, 56600),
 (1003, 19500),
 (1005, 15000),
 (1007, 22000),
 (1009, 15000),
 (1011, 65000)]

In [ ]:
completed_orders=reduced_orders.map(lambda x:(x[0],x[1]))
completed_orders.collect()

[(1001, 101),
 (1003, 101),
 (1005, 104),
 (1007, 102),
 (1008, 106),
 (1010, 108),
 (1011, 103)]

In [ ]:
result1=completed_orders.join(reduced_items)
result1.collect()

[(1008, (106, 12000)),
 (1001, (101, 56600)),
 (1005, (104, 15000)),
 (1010, (108, 10000)),
 (1003, (101, 19500)),
 (1007, (102, 22000)),
 (1011, (103, 65000))]

In [ ]:
updated_customer=customers_rdd.map(lambda x:(x[0],(x[1],x[2])))
updated_customer.collect()

[(101, ('Rahul', 'Delhi')),
 (102, ('Priya', 'Mumbai')),
 (103, ('Amit', 'Bangalore')),
 (104, ('Sneha', 'Hyderabad')),
 (105, ('Rohit', 'Chennai')),
 (106, ('Neha', 'Pune')),
 (107, ('Karan', 'Kolkata')),
 (108, ('Anjali', 'Bhubaneswar'))]

In [ ]:
updated_result1=result1.map(lambda x:(x[1][0],x[1][1]))
updated_result1.collect()

[(106, 12000),
 (101, 56600),
 (104, 15000),
 (108, 10000),
 (101, 19500),
 (102, 22000),
 (103, 65000)]

In [ ]:
result2=updated_customer.join(updated_result1)
result2.collect()

[(102, (('Priya', 'Mumbai'), 22000)),
 (108, (('Anjali', 'Bhubaneswar'), 10000)),
 (103, (('Amit', 'Bangalore'), 65000)),
 (104, (('Sneha', 'Hyderabad'), 15000)),
 (106, (('Neha', 'Pune'), 12000)),
 (101, (('Rahul', 'Delhi'), 56600)),
 (101, (('Rahul', 'Delhi'), 19500))]

In [ ]:
result3=result2.reduceByKey(lambda x,y:((x[0][0],x[0][1]),x[1]+y[1]))
result3.collect()

[(102, (('Priya', 'Mumbai'), 22000)),
 (108, (('Anjali', 'Bhubaneswar'), 10000)),
 (103, (('Amit', 'Bangalore'), 65000)),
 (104, (('Sneha', 'Hyderabad'), 15000)),
 (106, (('Neha', 'Pune'), 12000)),
 (101, (('Rahul', 'Delhi'), 76100))]

In [ ]:
result4=result3.filter(lambda x:x[1][1]>10000).sortBy(lambda x:x[1][1],ascending=False).take(5)
print("Final output:",result4)

Final output: [(101, (('Rahul', 'Delhi'), 76100)), (103, (('Amit', 'Bangalore'), 65000)), (102, (('Priya', 'Mumbai'), 22000)), (104, (('Sneha', 'Hyderabad'), 15000)), (106, (('Neha', 'Pune'), 12000))]
